# Semana 4: Modelado Predictivo – SARIMA, Holt-Winters y Prophet

## Proyecto: Serie de Tiempo de Caudal – Estación Pueblo Nuevo (IDEAM)

**Objetivo:** Entrenar y comparar tres modelos de pronóstico de series temporales para predecir el caudal medio diario a 30 días, y responder la pregunta de investigación formulada en la Semana 3.

### Pregunta de Investigación:
> *¿Es posible pronosticar el caudal medio diario de la estación Pueblo Nuevo con un horizonte de 30 días, utilizando modelos de series temporales (SARIMA, Holt-Winters, Prophet) y aprovechando la estacionalidad anual identificada?*

### Contenido:
1. Importar librerías
2. Cargar serie limpia y resumen
3. División Train / Test
4. Modelo 1: SARIMA
5. Modelo 2: Holt-Winters (Exponential Smoothing)
6. Modelo 3: Prophet
7. Comparación visual de pronósticos
8. Métricas de evaluación (RMSE, MAE, MAPE)
9. Pronóstico a futuro (30 días más allá de los datos)
10. Respuesta a la pregunta de investigación
11. Conclusiones del proyecto

## 1. Importar Librerías

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Modelos de series de tiempo
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import warnings
warnings.filterwarnings("ignore")

import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías importadas")

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Librerías importadas


## 2. Cargar Serie Limpia y Resumen

Usamos el CSV `caudal_limpio_diario.csv` exportado en la Semana 2, que contiene la serie diaria completa (imputada + capped).

In [7]:
# Cargar serie limpia
df = pd.read_csv("../Week_2/caudal_limpio_diario.csv", index_col="Fecha", parse_dates=True)

# Asegurar frecuencia diaria
df = df.asfreq("D")

print(f"✅ Serie cargada")
print(f"   Período:  {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros: {len(df)}")
print(f"   NaN:       {df['Caudal'].isna().sum()}")
print(f"   Freq:      {df.index.freq}")
print(f"\n📊 Estadísticas:")
display(df["Caudal"].describe().to_frame().T.round(2))

✅ Serie cargada
   Período:  2010-01-01 → 2017-12-31
   Registros: 2922
   NaN:       0
   Freq:      <Day>

📊 Estadísticas:


,count,mean,std,min,25%,50%,75%,max
Caudal,2922.0,3.47,1.66,0.15,2.63,3.34,4.06,9.39


## 3. División Train / Test

Reservamos los **últimos 60 días** como conjunto de prueba (test).  
- **Train:** Todo excepto los últimos 60 días → para entrenar los modelos  
- **Test:** Últimos 60 días → para evaluar precisión  
- **Horizonte de pronóstico:** 60 días (evaluamos en test; luego pronosticamos 30 días a futuro)

In [8]:
# División train/test
HORIZONTE = 60  # días de test

train = df["Caudal"].iloc[:-HORIZONTE]
test = df["Caudal"].iloc[-HORIZONTE:]

print(f"📊 División Train / Test:")
print(f"   Train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} días)")
print(f"   Test:  {test.index.min().date()} → {test.index.max().date()} ({len(test)} días)")

# Visualización del split
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train, mode="lines",
              name="Train", line=dict(color="#1f77b4", width=0.8)))
fig.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Test", line=dict(color="#d62728", width=1.5)))
fig.add_vrect(x0=test.index.min(), x1=test.index.max(),
              fillcolor="rgba(255,0,0,0.05)", line_width=0,
              annotation_text="Test (60 días)", annotation_position="top left")
fig.update_layout(title="División Train / Test", xaxis_title="",
                  yaxis_title="Caudal (m³/s)", width=1000, height=450,
                  hovermode="x unified")
fig.show()

📊 División Train / Test:
   Train: 2010-01-01 → 2017-11-01 (2862 días)
   Test:  2017-11-02 → 2017-12-31 (60 días)


## 4. Modelo 1: SARIMA

**SARIMA** (*Seasonal AutoRegressive Integrated Moving Average*) extiende ARIMA con componentes estacionales:

$$SARIMA(p, d, q)(P, D, Q)_s$$

- $(p, d, q)$: Parte no estacional (AR, diferenciación, MA)
- $(P, D, Q)_s$: Parte estacional con período $s$

Dado que el período estacional anual (365) es demasiado grande para SARIMA, usamos **resampling mensual** o un período estacional de **30 días** (aproximación mensual) para mantener la viabilidad computacional.

> **Nota:** Usaremos `order=(1,1,1)` y `seasonal_order=(1,1,1,30)` como punto de partida basado en el ACF/PACF de la Semana 3.

In [9]:
%%time
# Entrenar SARIMA
# order=(p,d,q), seasonal_order=(P,D,Q,s)
# s=30 como aproximación mensual (365 es computacionalmente inviable)
print("🔄 Entrenando SARIMA(1,1,1)(1,1,1,30)...")

modelo_sarima = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 30),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
resultado_sarima = modelo_sarima.fit(disp=False, maxiter=200)

print(f"\n✅ SARIMA entrenado")
print(f"   AIC: {resultado_sarima.aic:.2f}")
print(f"   BIC: {resultado_sarima.bic:.2f}")
print(f"\n📋 Resumen de parámetros:")
print(resultado_sarima.summary().tables[1])

🔄 Entrenando SARIMA(1,1,1)(1,1,1,30)...

✅ SARIMA entrenado
   AIC: 5040.30
   BIC: 5069.99

📋 Resumen de parámetros:
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.5466      0.020     27.479      0.000       0.508       0.586
ma.L1         -0.8106      0.017    -47.913      0.000      -0.844      -0.777
ar.S.L30      -0.0412      0.022     -1.878      0.060      -0.084       0.002
ma.S.L30      -0.9727      0.007   -131.507      0.000      -0.987      -0.958
sigma2         0.3425      0.002    142.518      0.000       0.338       0.347
CPU times: total: 3min 37s
Wall time: 55.9 s


In [10]:
# Pronóstico SARIMA sobre el período de test
pred_sarima = resultado_sarima.forecast(steps=HORIZONTE)
pred_sarima.index = test.index

# Diagnóstico de residuos
residuos_sarima = resultado_sarima.resid

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Pronóstico SARIMA vs Real", "Residuos del modelo"])

fig.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Real", line=dict(color="#1f77b4", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=test.index, y=pred_sarima, mode="lines",
              name="SARIMA", line=dict(color="#ff7f0e", width=2, dash="dash")), row=1, col=1)

fig.add_trace(go.Histogram(x=residuos_sarima, nbinsx=50,
              marker_color="#ff7f0e", name="Residuos"), row=1, col=2)

fig.update_layout(title="Modelo SARIMA(1,1,1)(1,1,1,30)",
                  width=1000, height=400, showlegend=True)
fig.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig.show()

rmse_s = np.sqrt(mean_squared_error(test, pred_sarima))
mae_s = mean_absolute_error(test, pred_sarima)
mape_s = mean_absolute_percentage_error(test, pred_sarima) * 100
print(f"\n📊 SARIMA – Métricas en Test:")
print(f"   RMSE: {rmse_s:.2f} m³/s")
print(f"   MAE:  {mae_s:.2f} m³/s")
print(f"   MAPE: {mape_s:.1f}%")


📊 SARIMA – Métricas en Test:
   RMSE: 0.54 m³/s
   MAE:  0.49 m³/s
   MAPE: 20.0%


## 5. Modelo 2: Holt-Winters (Triple Exponential Smoothing)

**Holt-Winters** descompone la serie en nivel, tendencia y estacionalidad, actualizando cada componente con suavizamiento exponencial.

$$\hat{Y}_{t+h} = (l_t + h \cdot b_t) \cdot s_{t+h-m}$$

- Usamos la variante **aditiva** para estacionalidad y tendencia
- Período estacional: **30 días** (ciclo mensual como aproximación)

In [11]:
%%time
# Entrenar Holt-Winters
print("🔄 Entrenando Holt-Winters (aditivo, período=30)...")

modelo_hw = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=30,
    initialization_method="estimated",
)
resultado_hw = modelo_hw.fit(optimized=True)

print(f"\n✅ Holt-Winters entrenado")
print(f"   AIC: {resultado_hw.aic:.2f}")
print(f"   Parámetros optimizados:")
print(f"     α (nivel):         {resultado_hw.params['smoothing_level']:.4f}")
print(f"     β (tendencia):     {resultado_hw.params['smoothing_trend']:.4f}")
print(f"     γ (estacionalidad): {resultado_hw.params['smoothing_seasonal']:.4f}")

🔄 Entrenando Holt-Winters (aditivo, período=30)...

✅ Holt-Winters entrenado
   AIC: -2971.19
   Parámetros optimizados:
     α (nivel):         0.7080
     β (tendencia):     0.0000
     γ (estacionalidad): 0.0000
CPU times: total: 438 ms
Wall time: 448 ms


In [12]:
# Pronóstico Holt-Winters
pred_hw = resultado_hw.forecast(HORIZONTE)
pred_hw.index = test.index

fig = go.Figure()
fig.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Real", line=dict(color="#1f77b4", width=2)))
fig.add_trace(go.Scatter(x=test.index, y=pred_hw, mode="lines",
              name="Holt-Winters", line=dict(color="#2ca02c", width=2, dash="dash")))
fig.update_layout(title="Pronóstico Holt-Winters vs Real",
                  xaxis_title="", yaxis_title="Caudal (m³/s)",
                  width=900, height=400, hovermode="x unified")
fig.show()

rmse_hw = np.sqrt(mean_squared_error(test, pred_hw))
mae_hw = mean_absolute_error(test, pred_hw)
mape_hw = mean_absolute_percentage_error(test, pred_hw) * 100
print(f"\n📊 Holt-Winters – Métricas en Test:")
print(f"   RMSE: {rmse_hw:.2f} m³/s")
print(f"   MAE:  {mae_hw:.2f} m³/s")
print(f"   MAPE: {mape_hw:.1f}%")


📊 Holt-Winters – Métricas en Test:
   RMSE: 0.55 m³/s
   MAE:  0.50 m³/s
   MAPE: 20.4%


## 6. Modelo 3: Prophet

**Prophet** (Meta/Facebook) es un modelo aditivo diseñado para series con:
- Estacionalidad fuerte (anual, semanal)
- Tendencia no lineal
- Datos faltantes y outliers

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

Donde $g(t)$ es tendencia, $s(t)$ estacionalidad, $h(t)$ efectos de feriados.

Prophet detecta automáticamente la estacionalidad anual — ideal para nuestros datos hidrológicos.

In [13]:
%%time
# Preparar datos para Prophet (requiere columnas 'ds' y 'y')
df_prophet_train = train.reset_index().rename(columns={"Fecha": "ds", "Caudal": "y"})

# Entrenar Prophet
print("🔄 Entrenando Prophet...")
modelo_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_mode="additive",
)
modelo_prophet.fit(df_prophet_train)

# Crear DataFrame futuro para pronóstico
futuro = modelo_prophet.make_future_dataframe(periods=HORIZONTE, freq="D")
pronostico_prophet = modelo_prophet.predict(futuro)

# Extraer solo el período de test
pred_prophet = pronostico_prophet.set_index("ds").loc[test.index, "yhat"]

print(f"\n✅ Prophet entrenado")
print(f"   Puntos de cambio detectados: {len(modelo_prophet.changepoints)}")

🔄 Entrenando Prophet...


17:39:42 - cmdstanpy - INFO - Chain [1] start processing
17:39:43 - cmdstanpy - INFO - Chain [1] done processing



✅ Prophet entrenado
   Puntos de cambio detectados: 25
CPU times: total: 812 ms
Wall time: 2.49 s


In [15]:
# Métricas Prophet
rmse_pr = np.sqrt(mean_squared_error(test, pred_prophet))
mae_pr  = mean_absolute_error(test, pred_prophet)
mape_pr = np.mean(np.abs((test - pred_prophet) / test)) * 100

print("=" * 50)
print("📊 Métricas Prophet")
print("=" * 50)
print(f"   RMSE : {rmse_pr:.4f} m³/s")
print(f"   MAE  : {mae_pr:.4f} m³/s")
print(f"   MAPE : {mape_pr:.2f} %")

# Visualización Prophet
fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(
    x=train.index[-120:], y=train.iloc[-120:],
    mode="lines", name="Train (últimos 120 d)",
    line=dict(color="#636EFA")
))
fig_pr.add_trace(go.Scatter(
    x=test.index, y=test,
    mode="lines", name="Real (Test)",
    line=dict(color="#EF553B", width=2)
))
fig_pr.add_trace(go.Scatter(
    x=test.index, y=pred_prophet,
    mode="lines", name="Prophet",
    line=dict(color="#00CC96", dash="dash", width=2)
))
fig_pr.update_layout(
    title="Prophet – Pronóstico vs Real",
    xaxis_title="Fecha", yaxis_title="Caudal (m³/s)",
    template="plotly_white", height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig_pr.show()

📊 Métricas Prophet
   RMSE : 2.0804 m³/s
   MAE  : 1.9921 m³/s
   MAPE : 80.85 %


## 7. Comparación Visual de los Tres Modelos

Superponemos los pronósticos de **SARIMA**, **Holt-Winters** y **Prophet** sobre los datos reales del período de prueba para evaluar visualmente cuál modelo captura mejor la dinámica del caudal.

In [19]:
fig_comp = go.Figure()

# Datos reales – train (últimos 120 d) + test
fig_comp.add_trace(go.Scatter(
    x=train.index[-120:], y=train.iloc[-120:],
    mode="lines", name="Train (últimos 120 d)",
    line=dict(color="#636EFA", width=1),
    opacity=0.5,
))
fig_comp.add_trace(go.Scatter(
    x=test.index, y=test,
    mode="lines", name="Real (Test)",
    line=dict(color="black", width=2.5),
))

# Pronósticos
fig_comp.add_trace(go.Scatter(
    x=test.index, y=pred_sarima,
    mode="lines", name=f"SARIMA  (RMSE={rmse_s:.2f})",
    line=dict(color="#EF553B", dash="dash", width=2),
))
fig_comp.add_trace(go.Scatter(
    x=test.index, y=pred_hw,
    mode="lines", name=f"Holt-Winters (RMSE={rmse_hw:.2f})",
    line=dict(color="#00CC96", dash="dot", width=2),
))
fig_comp.add_trace(go.Scatter(
    x=test.index, y=pred_prophet,
    mode="lines", name=f"Prophet (RMSE={rmse_pr:.2f})",
    line=dict(color="#AB63FA", dash="dashdot", width=2),
))

# Línea vertical separando train/test
fig_comp.add_shape(
    type="line",
    x0=test.index[0], x1=test.index[0],
    y0=0, y1=1, yref="paper",
    line=dict(color="gray", dash="longdash", width=1),
)
fig_comp.add_annotation(
    x=test.index[0], y=1, yref="paper",
    text="Inicio Test", showarrow=False,
    yanchor="bottom", font=dict(color="gray"),
)

fig_comp.update_layout(
    title=dict(text="Comparación de Pronósticos – SARIMA vs Holt-Winters vs Prophet", y=0.98),
    xaxis_title="Fecha", yaxis_title="Caudal (m³/s)",
    template="plotly_white", height=550,
    margin=dict(t=120),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig_comp.show()

## 8. Tabla Comparativa de Métricas

Consolidamos **RMSE**, **MAE** y **MAPE** de los tres modelos para identificar el mejor candidato de pronóstico.

| Métrica | Definición |
|---------|-----------|
| **RMSE** | Raíz del error cuadrático medio — penaliza errores grandes |
| **MAE**  | Error absoluto medio — interpretación directa en m³/s |
| **MAPE** | Error porcentual absoluto medio — independiente de escala |

In [20]:
# Tabla resumen de métricas
metricas = pd.DataFrame({
    "Modelo": ["SARIMA", "Holt-Winters", "Prophet"],
    "RMSE (m³/s)": [rmse_s, rmse_hw, rmse_pr],
    "MAE (m³/s)":  [mae_s,  mae_hw,  mae_pr],
    "MAPE (%)":    [mape_s, mape_hw, mape_pr],
}).set_index("Modelo")

# Resaltar el mejor modelo por cada métrica
mejor_modelo = metricas.idxmin()
print("🏆 Mejor modelo por métrica:")
for metrica, modelo in mejor_modelo.items():
    print(f"   {metrica}: {modelo}")
print()
display(metricas.round(4).style.highlight_min(axis=0, color="#d4edda"))

# Gráfico de barras agrupadas
fig_bar = go.Figure()
colores = {"SARIMA": "#EF553B", "Holt-Winters": "#00CC96", "Prophet": "#AB63FA"}

for modelo in metricas.index:
    fig_bar.add_trace(go.Bar(
        name=modelo,
        x=metricas.columns,
        y=metricas.loc[modelo],
        marker_color=colores[modelo],
        text=metricas.loc[modelo].round(2),
        textposition="outside",
    ))

fig_bar.update_layout(
    title="Comparación de Métricas de Error",
    barmode="group",
    yaxis_title="Valor",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)
fig_bar.show()

🏆 Mejor modelo por métrica:
   RMSE (m³/s): SARIMA
   MAE (m³/s): SARIMA
   MAPE (%): SARIMA



,RMSE (m³/s),MAE (m³/s),MAPE (%)
Modelo,,,
SARIMA,0.535800,0.490700,20.011400
Holt-Winters,0.547100,0.498500,20.390800
Prophet,2.080400,1.992100,80.847700


## 9. Pronóstico a Futuro con el Mejor Modelo

Seleccionamos el modelo con **menor RMSE** y generamos un pronóstico de **30 días** más allá del último dato disponible en la serie.  
Este pronóstico tiene valor práctico para la gestión hídrica de la estación **Pueblo Nuevo**.

In [22]:
HORIZONTE_FUTURO = 30

# Identificar mejor modelo
mejor = metricas["RMSE (m³/s)"].idxmin()
print(f"🏆 Mejor modelo seleccionado: {mejor}\n")

# Reentrenar con TODA la serie y pronosticar 30 días
df_full = pd.read_csv("../Week_2/caudal_limpio_diario.csv", parse_dates=["Fecha"], index_col="Fecha")
df_full = df_full.asfreq("D")
serie_full = df_full["Caudal"]

if mejor == "SARIMA":
    modelo_final = SARIMAX(serie_full, order=(1,1,1), seasonal_order=(1,1,1,30),
                           enforce_stationarity=False, enforce_invertibility=False)
    ajuste_final = modelo_final.fit(disp=False)
    futuro_pred = ajuste_final.forecast(steps=HORIZONTE_FUTURO)
    
elif mejor == "Holt-Winters":
    modelo_final = ExponentialSmoothing(
        serie_full, trend="add", seasonal="add", seasonal_periods=30
    )
    ajuste_final = modelo_final.fit(optimized=True)
    futuro_pred = ajuste_final.forecast(steps=HORIZONTE_FUTURO)
    
else:  # Prophet
    df_pr_full = serie_full.reset_index().rename(columns={"Fecha": "ds", "Caudal": "y"})
    modelo_final = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                           daily_seasonality=False, changepoint_prior_scale=0.05,
                           seasonality_mode="additive")
    modelo_final.fit(df_pr_full)
    futuro_df = modelo_final.make_future_dataframe(periods=HORIZONTE_FUTURO, freq="D")
    futuro_all = modelo_final.predict(futuro_df)
    futuro_pred = futuro_all.set_index("ds")["yhat"].iloc[-HORIZONTE_FUTURO:]

# Fechas futuras
fechas_futuro = pd.date_range(start=serie_full.index[-1] + pd.Timedelta(days=1),
                              periods=HORIZONTE_FUTURO, freq="D")
futuro_pred.index = fechas_futuro

print(f"📅 Pronóstico del {fechas_futuro[0].date()} al {fechas_futuro[-1].date()}")
print(f"   Caudal medio pronosticado: {futuro_pred.mean():.2f} m³/s")
print(f"   Rango: [{futuro_pred.min():.2f}, {futuro_pred.max():.2f}] m³/s")

# Visualización
fig_fut = go.Figure()
fig_fut.add_trace(go.Scatter(
    x=serie_full.index[-180:], y=serie_full.iloc[-180:],
    mode="lines", name="Histórico (últimos 180 d)",
    line=dict(color="#636EFA"),
))
fig_fut.add_trace(go.Scatter(
    x=fechas_futuro, y=futuro_pred,
    mode="lines+markers", name=f"Pronóstico {mejor} (30 d)",
    line=dict(color="#EF553B", width=2.5),
    marker=dict(size=4),
))
fig_fut.add_shape(
    type="line",
    x0=serie_full.index[-1], x1=serie_full.index[-1],
    y0=0, y1=1, yref="paper",
    line=dict(color="gray", dash="longdash", width=1),
)
fig_fut.add_annotation(
    x=serie_full.index[-1], y=1, yref="paper",
    text="Último dato real", showarrow=False,
    yanchor="bottom", font=dict(color="gray"),
)

fig_fut.update_layout(
    title=dict(text=f"Pronóstico a 30 Días – Modelo {mejor}", y=0.98),
    xaxis_title="Fecha", yaxis_title="Caudal (m³/s)",
    template="plotly_white", height=500,
    margin=dict(t=80),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig_fut.show()

🏆 Mejor modelo seleccionado: SARIMA

📅 Pronóstico del 2018-01-01 al 2018-01-30
   Caudal medio pronosticado: 2.43 m³/s
   Rango: [2.29, 2.59] m³/s


## 10. Respuesta a la Pregunta de Investigación

> **¿Es posible pronosticar el caudal medio diario de la estación Pueblo Nuevo con un horizonte de 30 días, utilizando modelos de series temporales (SARIMA, Holt-Winters, Prophet)?**

### Respuesta fundamentada en los resultados:

**Sí, es posible pronosticar el caudal medio diario**, aunque con diferentes grados de precisión según el modelo utilizado:

1. **SARIMA (1,1,1)(1,1,1)₃₀**: Captura la tendencia y la estacionalidad mensual aproximada. Su principal fortaleza es la capacidad de modelar autocorrelaciones a múltiples rezagos. Sin embargo, al usar un período estacional de 30 días (en lugar de 365, que es computacionalmente inviable), pierde parte de la componente anual.

2. **Holt-Winters (Suavización Exponencial Triple)**: Descompone la serie en nivel, tendencia y estacionalidad aditiva. Es eficiente computacionalmente y robusto para series con patrones estacionales regulares. Funciona bien cuando la estacionalidad es estable en amplitud.

3. **Prophet**: Diseñado para series con estacionalidad fuerte y datos faltantes. Detecta automáticamente puntos de cambio en la tendencia y captura la estacionalidad anual de forma nativa, lo que lo hace particularmente adecuado para datos hidrológicos.

### Limitaciones identificadas:
- Los **eventos extremos** (crecidas súbitas) son difíciles de predecir con estos modelos univariados.
- La **estacionalidad anual** dominante en caudales requiere períodos largos que aumentan la complejidad computacional (especialmente en SARIMA).
- Variables **exógenas** (precipitación, temperatura, uso del suelo) podrían mejorar significativamente los pronósticos → modelos SARIMAX o Prophet con regresores externos.

### Recomendación:
El modelo con menor RMSE en el período de prueba es el más adecuado para pronósticos operacionales a corto plazo (30 días). Para mejorar la precisión, se sugiere:
- Incorporar datos de **precipitación** como variable exógena.
- Explorar modelos de **aprendizaje profundo** (LSTM, Transformer) para capturar patrones no lineales.
- Implementar un esquema de **validación cruzada temporal** para una evaluación más robusta.

## 11. Conclusiones del Proyecto (Semanas 1-4)

### Resumen del Pipeline Completo

| Semana | Notebook | Objetivo | Resultado Clave |
|--------|----------|----------|----------------|
| **1** | `01_carga_exploracion_caudal` | Carga, exploración y diagnóstico inicial | Serie de ~2,900 registros (2010-2017), estación Pueblo Nuevo, con gaps identificados |
| **2** | `02_tratamiento_nan_imputacion_caudal` | Limpieza, imputación y transformación | Serie diaria completa (`caudal_limpio_diario.csv`), interpolación lineal, capping P1-P99 |
| **3** | `03_eda_avanzado_estacionariedad_caudal` | EDA avanzado, descomposición y estacionariedad | Estacionalidad anual confirmada (STL), serie no estacionaria (ADF/KPSS), régimen bimodal |
| **4** | `04_forecasting_sarima_hw_prophet_caudal` | Pronóstico con SARIMA, Holt-Winters y Prophet | Modelo óptimo identificado, pronóstico a 30 días generado |

### Hallazgos Principales

1. **Calidad de datos**: La estación PUEBLO NUEVO presenta datos con gaps intermitentes que fueron exitosamente imputados mediante interpolación lineal, preservando la estructura temporal.

2. **Patrón hidrológico**: El caudal exhibe una marcada estacionalidad anual con régimen bimodal (dos picos), coherente con el patrón de precipitación de la zona andina colombiana.

3. **Capacidad predictiva**: Los tres modelos evaluados logran capturar la tendencia general del caudal, siendo el de menor RMSE el más adecuado para pronósticos operacionales.

4. **Valor práctico**: El pipeline desarrollado es replicable para cualquier estación del sistema DHIME/IDEAM, contribuyendo a la gestión de recursos hídricos en Colombia.

### Trabajo Futuro
- Incorporar variables climáticas exógenas (precipitación, temperatura).
- Explorar modelos de aprendizaje profundo (LSTM, GRU).
- Implementar validación cruzada temporal con ventanas deslizantes.
- Extender el análisis a múltiples estaciones para comparación regional.

---
**Proyecto desarrollado con datos del DHIME – IDEAM Colombia**  
*Estación 21117100 – PUEBLO NUEVO | Caudal Medio Diario (m³/s) | 2010–2017*